# Tarea 12 — Simulación del flujo temporal

**Proyecto Integrador — Minería de Datos Masivos**

En esta etapa se transforman los datos históricos de contrataciones públicas en secuencias ordenadas temporalmente para simular la llegada de eventos como un flujo.

Se construirán dos flujos:

1. **Flujo de procesos:** un evento por proceso de contratación. Será utilizado posteriormente por DGIM.
2. **Flujo de participaciones de postores:** un evento por postor y proceso. Será utilizado posteriormente por Bloom Filter y Count-Min Sketch.

La variable temporal utilizada es `fecha_pub`.

In [1]:
from pathlib import Path

import pandas as pd

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SEACE-Flujos")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Paralelismo:", spark.sparkContext.defaultParallelism)

26/10/04 18:03:30 WARN Utils: Your hostname, willian-pc resolves to a loopback address: 127.0.1.1; using 192.168.18.28 instead (on interface wlo1)
26/10/04 18:03:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/04 18:03:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/04 18:03:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark: 3.5.3
Paralelismo: 8


In [3]:
PROJECT_ROOT = Path.home() / "dm"
DATA_DIR = PROJECT_ROOT / "parquet"

procesos = spark.read.parquet(str(DATA_DIR / "procesos"))
postores = spark.read.parquet(str(DATA_DIR / "postores"))

print("Procesos:", procesos.count())
print("Postores:", postores.count())

Procesos: 209740
Postores: 2493612


In [4]:
stream_procesos = (
    procesos
    .select(
        "fecha_pub",
        "release_id",
        "estado_proceso"
    )
    .withColumn(
        "es_desierto",
        (F.col("estado_proceso") == "DESIERTO").cast("int")
    )
    .orderBy(
        "fecha_pub",
        "release_id"
    )
)

In [5]:
print("Eventos del flujo:", stream_procesos.count())

Eventos del flujo: 209740


In [6]:
stream_procesos.select(
    F.sum(F.col("fecha_pub").isNull().cast("int")).alias("fecha_pub_nulos"),
    F.sum(F.col("release_id").isNull().cast("int")).alias("release_id_nulos"),
    F.sum(F.col("estado_proceso").isNull().cast("int")).alias("estado_nulos")
).show()

+---------------+----------------+------------+
|fecha_pub_nulos|release_id_nulos|estado_nulos|
+---------------+----------------+------------+
|              0|               0|           0|
+---------------+----------------+------------+



In [7]:
stream_procesos.show(10, truncate=False)

+-------------------+-------------------------------------------------------------------+--------------+-----------+
|fecha_pub          |release_id                                                         |estado_proceso|es_desierto|
+-------------------+-------------------------------------------------------------------+--------------+-----------+
|2024-01-02 13:14:00|ocds-dgv273-seacev3-984421-2024-02-15T08:17:02.457498-05:00        |CONTRATADO    |0          |
|2024-01-02 21:17:00|ocds-dgv273-seacev3-984400-2026-02-01T09:56:02.910937-05:00        |CONTRATADO    |0          |
|2024-01-03 08:58:00|ocds-dgv273-seacev3-984361-2025-06-24T17:01:18.172950-05:00        |CONTRATADO    |0          |
|2024-01-03 10:53:00|ocds-dgv273-seacev3-984402-2026-02-26T10:08:45.554222-05:00        |CONTRATADO    |0          |
|2024-01-03 16:39:00|ocds-dgv273-seacev3-984441-2024-02-11T08:11:56.741918-05:00        |CONTRATADO    |0          |
|2024-01-03 19:06:00|ocds-dgv273-seacev3-984466-2026-02-06T10:03

In [8]:
stream_procesos.orderBy(
    F.desc("fecha_pub"),
    F.desc("release_id")
).show(10, truncate=False)

+-------------------+------------------------------------------------------------+--------------+-----------+
|fecha_pub          |release_id                                                  |estado_proceso|es_desierto|
+-------------------+------------------------------------------------------------+--------------+-----------+
|2026-08-31 23:55:00|ocds-dgv273-seacev3-1246170-2026-09-03T10:50:03.458988-05:00|CONVOCADO     |0          |
|2026-08-31 23:53:00|ocds-dgv273-seacev3-1246172-2026-09-03T10:50:38.750750-05:00|CONVOCADO     |0          |
|2026-08-31 23:52:00|ocds-dgv273-seacev3-1246171-2026-09-03T10:50:40.359322-05:00|CONVOCADO     |0          |
|2026-08-31 23:52:00|ocds-dgv273-seacev3-1245761-2026-09-03T10:49:58.546741-05:00|CONVOCADO     |0          |
|2026-08-31 23:39:00|ocds-dgv273-seacev3-1246182-2026-09-03T10:49:57.711373-05:00|CONVOCADO     |0          |
|2026-08-31 23:38:00|ocds-dgv273-seacev3-1246133-2026-09-03T10:50:50.210966-05:00|CONVOCADO     |0          |
|2026-08-3

In [9]:
stream_procesos.groupBy(
    "estado_proceso",
    "es_desierto"
).count().orderBy(
    F.desc("count")
).show(30, truncate=False)

+--------------------------------------------------+-----------+------+
|estado_proceso                                    |es_desierto|count |
+--------------------------------------------------+-----------+------+
|CONTRATADO                                        |0          |131154|
|DESIERTO                                          |1          |25268 |
|CONVOCADO                                         |0          |13577 |
|NULO                                              |0          |11874 |
|CONSENTIDO                                        |0          |10700 |
|MIXTO                                             |0          |6721  |
|ADJUDICADO                                        |0          |3395  |
|RETROTRAIDO_POR_RESOLUCION                        |0          |3150  |
|CANCELADO                                         |0          |2812  |
|PENDIENTE_DE_REGISTRO_DE_EFECTO                   |0          |280   |
|APELADO                                           |0          |

In [10]:
w_orden = Window.orderBy(
    "fecha_pub",
    "release_id"
)

validacion_orden = (
    stream_procesos
    .withColumn(
        "fecha_anterior",
        F.lag("fecha_pub").over(w_orden)
    )
    .withColumn(
        "orden_invalido",
        (
            F.col("fecha_anterior").isNotNull()
            & (F.col("fecha_pub") < F.col("fecha_anterior"))
        ).cast("int")
    )
)

In [11]:
validacion_orden.select(
    F.sum("orden_invalido").alias("eventos_fuera_de_orden")
).show()

+----------------------+
|eventos_fuera_de_orden|
+----------------------+
|                     0|
+----------------------+



In [12]:
empates_fecha = (
    procesos
    .groupBy("fecha_pub")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Instantes con más de un proceso:",
    empates_fecha.count()
)

empates_fecha.orderBy(
    F.desc("count")
).show(10, truncate=False)

Instantes con más de un proceso: 40122
+-------------------+-----+
|fecha_pub          |count|
+-------------------+-----+
|2025-04-21 23:59:00|19   |
|2025-04-21 22:58:00|16   |
|2025-04-21 23:35:00|16   |
|2025-04-21 21:55:00|13   |
|2025-04-21 21:35:00|13   |
|2025-04-21 21:45:00|13   |
|2025-04-21 23:11:00|12   |
|2025-04-21 22:53:00|12   |
|2025-04-21 22:01:00|11   |
|2025-04-21 23:07:00|11   |
+-------------------+-----+
only showing top 10 rows



In [13]:
stream_procesos.select(
    F.min("fecha_pub").alias("primer_evento"),
    F.max("fecha_pub").alias("ultimo_evento"),
    F.count("*").alias("n_eventos"),
    F.sum("es_desierto").alias("n_desiertos")
).show(truncate=False)

+-------------------+-------------------+---------+-----------+
|primer_evento      |ultimo_evento      |n_eventos|n_desiertos|
+-------------------+-------------------+---------+-----------+
|2024-01-02 13:14:00|2026-08-31 23:55:00|209740   |25268      |
+-------------------+-------------------+---------+-----------+



### Validación del flujo de procesos

El flujo contiene 209,740 eventos ordenados cronológicamente entre el 2 de enero de 2024 y el 31 de agosto de 2026.

No existen valores nulos en la fecha, el identificador del proceso ni el estado. Se identificaron 25,268 procesos desiertos, que se representan mediante `es_desierto = 1`.

Existen múltiples procesos publicados exactamente en el mismo instante; por ello, se utiliza `release_id` como criterio secundario de ordenamiento para obtener una secuencia determinista.

La validación confirmó que no existen eventos fuera del orden temporal definido.

## Flujo de participaciones de postores

Para Bloom Filter y Count-Min Sketch se construye un segundo flujo en el que cada evento representa la participación de un postor en un proceso de contratación.

La fecha del evento corresponde a `fecha_pub` del proceso. Cada evento contiene:

- fecha de publicación;
- identificador del proceso;
- RUC del postor;
- indicador de postor extranjero.

El flujo se ordena por `fecha_pub`, `release_id` y `ruc` para obtener una secuencia determinista.

In [14]:
stream_postores = (
    postores
    .join(
        procesos.select(
            "release_id",
            "fecha_pub"
        ),
        on="release_id",
        how="inner"
    )
    .select(
        "fecha_pub",
        "release_id",
        "ruc",
        "extranjero"
    )
    .orderBy(
        "fecha_pub",
        "release_id",
        "ruc"
    )
)

In [15]:
print("Eventos del flujo de postores:", stream_postores.count())

Eventos del flujo de postores: 2493612


In [16]:
stream_postores.select(
    F.sum(F.col("fecha_pub").isNull().cast("int")).alias("fecha_pub_nulos"),
    F.sum(F.col("release_id").isNull().cast("int")).alias("release_id_nulos"),
    F.sum(F.col("ruc").isNull().cast("int")).alias("ruc_nulos"),
    F.sum(F.col("extranjero").isNull().cast("int")).alias("extranjero_nulos")
).show()

+---------------+----------------+---------+----------------+
|fecha_pub_nulos|release_id_nulos|ruc_nulos|extranjero_nulos|
+---------------+----------------+---------+----------------+
|              0|               0|        0|               0|
+---------------+----------------+---------+----------------+



In [17]:
postores_huerfanos = (
    postores
    .join(
        procesos.select("release_id"),
        on="release_id",
        how="left_anti"
    )
)

print("Postores sin proceso asociado:", postores_huerfanos.count())

Postores sin proceso asociado: 0


In [18]:
stream_postores.show(10, truncate=False)

[Stage 51:=============================>                            (4 + 4) / 8]

+-------------------+-------------------------------------------------------------------+-----------+----------+
|fecha_pub          |release_id                                                         |ruc        |extranjero|
+-------------------+-------------------------------------------------------------------+-----------+----------+
|2024-01-02 13:14:00|ocds-dgv273-seacev3-984421-2024-02-15T08:17:02.457498-05:00        |20600123166|false     |
|2024-01-02 21:17:00|ocds-dgv273-seacev3-984400-2026-02-01T09:56:02.910937-05:00        |10316****22|false     |
|2024-01-03 08:58:00|ocds-dgv273-seacev3-984361-2025-06-24T17:01:18.172950-05:00        |20602005021|false     |
|2024-01-03 10:53:00|ocds-dgv273-seacev3-984402-2026-02-26T10:08:45.554222-05:00        |20515703595|false     |
|2024-01-03 16:39:00|ocds-dgv273-seacev3-984441-2024-02-11T08:11:56.741918-05:00        |20545527180|false     |
|2024-01-03 19:06:00|ocds-dgv273-seacev3-984466-2026-02-06T10:03:33.882971-05:00        |2060039

In [19]:
n_eventos_postores = stream_postores.count()

n_eventos_unicos = (
    stream_postores
    .select("release_id", "ruc")
    .dropDuplicates()
    .count()
)

print("Eventos:", n_eventos_postores)
print("Pares release_id-ruc únicos:", n_eventos_unicos)
print("Duplicados:", n_eventos_postores - n_eventos_unicos)

[Stage 57:>                                                         (0 + 8) / 8]

Eventos: 2493612
Pares release_id-ruc únicos: 2493612
Duplicados: 0


In [20]:
stream_postores.select(
    F.count("*").alias("n_eventos"),
    F.countDistinct("ruc").alias("ruc_distintos"),
    F.sum(F.col("extranjero").cast("int")).alias("eventos_extranjeros")
).show()

[Stage 64:>                                                         (0 + 8) / 8]

+---------+-------------+-------------------+
|n_eventos|ruc_distintos|eventos_extranjeros|
+---------+-------------+-------------------+
|  2493612|       226293|               4611|
+---------+-------------+-------------------+



In [21]:
frecuencia_ruc = (
    stream_postores
    .groupBy("ruc")
    .agg(
        F.count("*").alias("frecuencia")
    )
    .orderBy(F.desc("frecuencia"))
)

frecuencia_ruc.show(10, truncate=False)

[Stage 73:>                                                         (0 + 7) / 7]

+-----------+----------+
|ruc        |frecuencia|
+-----------+----------+
|20611499591|9943      |
|20605753907|9100      |
|20613035398|6936      |
|20612573621|6700      |
|20541777980|6400      |
|20542513439|6240      |
|20603425104|5921      |
|10283****79|5684      |
|20613921002|5369      |
|20601206481|5251      |
+-----------+----------+
only showing top 10 rows



In [22]:
stream_postores.select(
    F.min("fecha_pub").alias("primer_evento"),
    F.max("fecha_pub").alias("ultimo_evento"),
    F.count("*").alias("n_eventos"),
    F.countDistinct("ruc").alias("ruc_distintos")
).show(truncate=False)

[Stage 75:>                                                         (0 + 8) / 8]

+-------------------+-------------------+---------+-------------+
|primer_evento      |ultimo_evento      |n_eventos|ruc_distintos|
+-------------------+-------------------+---------+-------------+
|2024-01-02 13:14:00|2026-08-31 23:00:00|2493612  |226293       |
+-------------------+-------------------+---------+-------------+



## Conclusión de la simulación del flujo

Se construyeron y validaron dos secuencias temporales:

- **Flujo de procesos:** 209,740 eventos, utilizado posteriormente para DGIM.
- **Flujo de participaciones de postores:** 2,493,612 eventos correspondientes a 226,293 RUC distintos, utilizado para Bloom Filter y Count-Min Sketch.

Ambos flujos están ordenados mediante `fecha_pub` y utilizan criterios secundarios deterministas para resolver eventos que comparten la misma marca temporal.

No se encontraron fechas, identificadores ni RUC nulos, tampoco participaciones sin proceso asociado ni pares `release_id-ruc` duplicados.

La construcción permite reutilizar una misma secuencia de participaciones para dos problemas diferentes:
- Bloom Filter: detectar si un RUC ya apareció previamente.
- Count-Min Sketch: estimar cuántas participaciones acumula un RUC.